# Visualize Prediction Results

Two-stage visualization pipeline:
1. **Aggregation** — sums numeric-named GeoTIFFs across per-run subfolders into a single `summed/` folder per model×modality combination (preserving CRS/transform via rasterio).
2. **Plotting** — produces a 3×4 figure per frame: rows = modalities (AE, PS, S2); columns = UNET, SWIN, TERRAMIND, LABEL.

Set all paths in the **Configuration** cell before running.

In [ ]:
%matplotlib inline
from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff

try:
    import rasterio
except ImportError as e:
    raise ImportError(
        "rasterio is required. Install with: pip install rasterio"
    ) from e

try:
    from tqdm.auto import tqdm
except ImportError as e:
    raise ImportError("tqdm is required. Install with: pip install tqdm") from e

## Configuration

Edit paths, VIZ_NAMES, and style settings below.

In [ ]:
# -------------------------
# Result directories
# -------------------------
RESULTS_DIR = r"N:\isipd\projects\p_planetdw\data\methods_test\results"

UNET_AE_DIR = os.path.join(RESULTS_DIR, r"UNET\AE")
UNET_PS_DIR = os.path.join(RESULTS_DIR, r"UNET\PS")
UNET_S2_DIR = os.path.join(RESULTS_DIR, r"UNET\S2")

SWIN_AE_DIR = os.path.join(RESULTS_DIR, r"Swin\AE")
SWIN_PS_DIR = os.path.join(RESULTS_DIR, r"Swin\PS")
SWIN_S2_DIR = os.path.join(RESULTS_DIR, r"Swin\S2")

TERRAMIND_AE_DIR = os.path.join(RESULTS_DIR, r"TERRAMIND\AE")
TERRAMIND_PS_DIR = os.path.join(RESULTS_DIR, r"TERRAMIND\PS")
TERRAMIND_S2_DIR = os.path.join(RESULTS_DIR, r"TERRAMIND\S2")

TRAINING_IMAGE_DIR_AE = r"N:\isipd\projects\p_planetdw\data\methods_test\preprocessed\AE"
TRAINING_IMAGE_DIR_PS = r"N:\isipd\projects\p_planetdw\data\methods_test\preprocessed\20260109-1434_UNETxPS"
TRAINING_IMAGE_DIR_S2 = r"N:\isipd\projects\p_planetdw\data\methods_test\preprocessed\20260108-1335_UNETxS2"

# Frame IDs to visualize
VIZ_NAMES = [
    "34","60","6","173","63","74","218","126","347","325","167","290","36","123",
    "324","193","236","182","72","367","220","183","25","340","132","56","238",
    "113","85","245","242","42","213","55","209","386","84","97","64","131",
    "354","272","357","111","164","109","375","346","5","11","196","89","119",
    "259","200","273","199","32","395","298","234","394","227","302","388","289",
    "293","158","237","360","390","291","296","189","161","153","83","70","23","239"
]

# -------------------------
# Style settings
# -------------------------
OVERLAY_CMAP      = "cividis"   # also: "magma" "plasma" "inferno"
OVERLAY_VMIN      = 0.0
OVERLAY_VMAX      = 10.0
OVERLAY_CLIP      = True

# Opacity mask: counts > eps => fully opaque; counts <= eps => transparent
OVERLAY_OPAQUE_EPS = 0.0

ADD_CONTOURS      = False

LABEL_THRESHOLD   = 0.0
LABEL_ALPHA       = 0.75

DEBUG_OVERLAY_STATS = False
DEBUG_LABEL_STATS   = False

## Helper Functions

In [ ]:
def _find_tiff_by_stem(folder: Path, stem: str) -> Optional[Path]:
    for ext in (".tif", ".tiff", ".TIF", ".TIFF"):
        p = folder / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def _bands_first(arr: np.ndarray) -> bool:
    """Heuristic: (bands, H, W) vs (H, W, bands)."""
    if arr.ndim != 3:
        raise ValueError("Expected 3D array.")
    return arr.shape[0] <= 16 and arr.shape[0] < arr.shape[1] and arr.shape[0] < arr.shape[2]


def _normalize_rgb(rgb: np.ndarray) -> np.ndarray:
    """Percentile stretch to 0..1 per channel (NaN-safe)."""
    rgb = rgb.astype(np.float32, copy=False)
    out = np.empty_like(rgb, dtype=np.float32)
    for c in range(3):
        ch = rgb[..., c]
        lo = np.nanpercentile(ch, 2)
        hi = np.nanpercentile(ch, 98)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            out[..., c] = 0.0
        else:
            out[..., c] = np.clip((ch - lo) / (hi - lo), 0, 1)
    return out


def _rgb_indices_for_method(method: str) -> Tuple[int, int, int]:
    """
    Returns (B_idx, G_idx, R_idx) in the band axis.
      AE/PS: B,G,R,...,Label => (0,1,2)
      S2:    Coastal,B,G,R,...,Label => (1,2,3)
    """
    method = method.upper()
    if method in ("AE", "PS"):
        return (0, 1, 2)
    if method == "S2":
        return (1, 2, 3)
    raise ValueError(f"Unknown method: {method}")


def _aggregation_done_for_names(root: Path, names: List[str],
                                 summed_folder_name: str = "summed") -> bool:
    """Aggregation considered done if root/summed has files for all names."""
    summed_dir = root / summed_folder_name
    if not summed_dir.exists() or not summed_dir.is_dir():
        return False
    for nm in names:
        if not str(nm).isdigit():
            continue
        if _find_tiff_by_stem(summed_dir, str(nm)) is None:
            return False
    return True

## GeoTIFF Aggregation

In [ ]:
def sum_numeric_named_geotiffs_across_subfolders(
    root_dir,
    output_folder_name: str = "summed",
    *,
    require_all_subfolders: bool = True,
    overwrite: bool = True,
    keep_bands: bool = False,
    dtype_out: str = "float32",
    compress: str = "DEFLATE",
    show_progress: bool = True,
    verbose: bool = True,
) -> Path:
    root = Path(root_dir).resolve()
    out_dir = root / output_folder_name
    out_dir.mkdir(parents=True, exist_ok=True)

    if verbose:
        print(f"\n[SUM-GEO] Root: {root}")

    if not root.exists():
        raise FileNotFoundError(f"Missing root: {root}")

    subfolders = [p for p in root.iterdir() if p.is_dir() and p != out_dir]

    if not subfolders:
        direct_numeric = [
            f for f in root.iterdir()
            if f.is_file() and f.suffix.lower() in (".tif", ".tiff") and f.stem.isdigit()
        ]
        if direct_numeric:
            if verbose:
                print(f"[SUM-GEO][WARN] No subfolders found; treating root as already final.")
            return out_dir
        raise ValueError(f"No subfolders found under: {root}")

    if verbose:
        print(f"[SUM-GEO] Found {len(subfolders)} subfolders.")

    per_folder: List[Dict[str, Path]] = []
    for sf in tqdm(subfolders, desc=f"Scanning ({root.name})", disable=not show_progress):
        m: Dict[str, Path] = {}
        for f in sf.iterdir():
            if f.is_file() and f.suffix.lower() in (".tif", ".tiff") and f.stem.isdigit():
                m[f.stem] = f
        per_folder.append(m)

    if require_all_subfolders:
        common = set(per_folder[0].keys())
        for m in per_folder[1:]:
            common &= set(m.keys())
        names = sorted(common, key=lambda s: int(s))
    else:
        union = set()
        for m in per_folder:
            union |= set(m.keys())
        names = sorted(union, key=lambda s: int(s))

    if not names:
        raise ValueError(f"No numeric TIFF names found under {root}.")

    written = 0
    skipped = 0

    for name in tqdm(names, desc=f"Summing ({root.name})", disable=not show_progress):
        out_path = out_dir / f"{name}.tif"
        if out_path.exists() and not overwrite:
            skipped += 1
            continue

        acc = None
        ref_profile = None
        used_files = 0

        for m in per_folder:
            if name not in m:
                if require_all_subfolders:
                    raise FileNotFoundError(f"Missing {name}.tif in one subfolder under {root}")
                else:
                    continue

            p = m[name]
            with rasterio.open(p) as src:
                data = src.read(masked=True).filled(0)
                data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

                if ref_profile is None:
                    ref_profile = src.profile.copy()

                if not keep_bands and data.shape[0] > 1:
                    data = data.sum(axis=0, keepdims=True)

                if acc is None:
                    acc = np.zeros_like(data, dtype=np.float64)

                acc += data.astype(np.float64, copy=False)
                used_files += 1

        if acc is None or ref_profile is None:
            continue

        out_profile = ref_profile.copy()
        out_profile.update(
            driver="GTiff", dtype=dtype_out, count=acc.shape[0],
            compress=compress, nodata=None,
        )

        with rasterio.open(out_path, "w", **out_profile) as dst:
            dst.write(acc.astype(dtype_out, copy=False))

        written += 1

    if verbose:
        print(f"[SUM-GEO] Done. Wrote {written} GeoTIFF(s), skipped {skipped} -> {out_dir}")

    return out_dir

## Image Loading & Overlay Drawing

In [ ]:
def load_training_rgb_and_label(training_tif: Path, method: str) -> Tuple[np.ndarray, np.ndarray]:
    """Load preprocessed training image -> rgb (H,W,3) + label (H,W, last band)."""
    arr = tiff.imread(str(training_tif))
    if arr.ndim != 3:
        raise ValueError(f"Training image must be multi-band: {training_tif}")

    if _bands_first(arr):
        label = arr[-1]
        b_idx, g_idx, r_idx = _rgb_indices_for_method(method)
        B, G, R = arr[b_idx], arr[g_idx], arr[r_idx]
    else:
        label = arr[..., -1]
        b_idx, g_idx, r_idx = _rgb_indices_for_method(method)
        B, G, R = arr[..., b_idx], arr[..., g_idx], arr[..., r_idx]

    label = np.nan_to_num(label, nan=0.0, posinf=0.0, neginf=0.0)
    rgb = _normalize_rgb(np.stack([R, G, B], axis=-1))
    return rgb, label


def load_overlay_counts_0_10(overlay_path: Path) -> np.ndarray:
    """Load overlay as raw counts clipped to [0, OVERLAY_VMAX]."""
    with rasterio.open(overlay_path) as src:
        data = src.read(masked=True).filled(0)

    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    x = data.sum(axis=0) if data.shape[0] > 1 else data[0]
    x = x.astype(np.float32, copy=False)

    if OVERLAY_CLIP:
        x = np.clip(x, OVERLAY_VMIN, OVERLAY_VMAX)

    return x


def draw_overlay_counts(ax, counts: np.ndarray, *, add_contours: bool = True):
    """Draw counts with binary opacity (0 => transparent, >0 => opaque)."""
    alpha_mask = (counts > OVERLAY_OPAQUE_EPS).astype(np.float32)

    im = ax.imshow(
        counts, cmap=OVERLAY_CMAP, alpha=alpha_mask,
        vmin=OVERLAY_VMIN, vmax=OVERLAY_VMAX,
    )

    if add_contours:
        levels = [lv for lv in [1, 2, 4, 6, 8, 10] if OVERLAY_VMIN < lv < OVERLAY_VMAX]
        if levels:
            ax.contour(counts, levels=levels, colors="white", linewidths=0.7, alpha=0.9)

    return im

## 3×4 Plotting Function

In [ ]:
def plot_3x4_overlays_with_label_col(
    name: str,
    training_dirs: Dict[str, Path],
    summed_dirs: Dict[Tuple[str, str], Path],
    out_dir,
    *,
    verbose: bool = True,
    add_contours: bool = True,
) -> Path:
    name = str(name)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    models  = ["UNET", "SWIN", "TERRAMIND"]
    methods = ["AE", "PS", "S2"]
    col_titles = ["UNET", "SWIN", "TERRAMIND", "LABEL (bin)"]

    row_rgb: Dict[str, Optional[np.ndarray]] = {}
    row_label: Dict[str, Optional[np.ndarray]] = {}

    for method in methods:
        tdir = training_dirs[method]
        tpath = _find_tiff_by_stem(tdir, name)
        if tpath is None:
            row_rgb[method] = row_label[method] = None
            continue
        try:
            rgb, lab = load_training_rgb_and_label(tpath, method=method)
            row_rgb[method] = rgb
            row_label[method] = lab
        except Exception as ex:
            row_rgb[method] = row_label[method] = None
            if verbose:
                print(f"[PLOT][WARN] Failed to load for method={method}, name={name}: {ex}")

    fig, axes = plt.subplots(3, 4, figsize=(16, 12), constrained_layout=True)
    colorbar_im = None

    for r, method in enumerate(methods):
        bg  = row_rgb[method]
        lab = row_label[method]

        for c, model in enumerate(models):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])

            if bg is None:
                ax.text(0.5, 0.5, "missing\ntraining",
                        transform=ax.transAxes, ha="center", va="center", fontsize=12,
                        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"))
            else:
                ax.imshow(bg)

            root = summed_dirs[(model, method)]
            summed_folder = root / "summed"
            overlay_path = _find_tiff_by_stem(summed_folder, name) if summed_folder.exists() else None
            if overlay_path is None:
                overlay_path = _find_tiff_by_stem(root, name)

            if overlay_path is None:
                ax.text(0.5, 0.12, "missing\noverlay",
                        transform=ax.transAxes, ha="center", va="center", fontsize=10,
                        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
            else:
                try:
                    counts = load_overlay_counts_0_10(overlay_path)
                    im = draw_overlay_counts(ax, counts, add_contours=add_contours)
                    if colorbar_im is None:
                        colorbar_im = im
                except Exception as ex:
                    ax.text(0.5, 0.12, "overlay\nerror",
                            transform=ax.transAxes, ha="center", va="center", fontsize=10,
                            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
                    if verbose:
                        print(f"[PLOT][WARN] Overlay load failed for {model}/{method}, name={name}: {ex}")

            if r == 0: ax.set_title(col_titles[c], fontsize=14)
            if c == 0: ax.set_ylabel(method, fontsize=14)

        # Label column
        axL = axes[r, 3]
        axL.set_xticks([]); axL.set_yticks([])

        if bg is None or lab is None:
            axL.text(0.5, 0.5, "missing\nlabel",
                     transform=axL.transAxes, ha="center", va="center", fontsize=12,
                     bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"))
        else:
            axL.imshow(bg)
            lab_bin = (lab > LABEL_THRESHOLD).astype(np.uint8)
            axL.imshow(lab_bin, cmap="viridis", vmin=0, vmax=1,
                       alpha=LABEL_ALPHA, interpolation="nearest")

        if r == 0: axL.set_title(col_titles[3], fontsize=14)

    fig.suptitle(f"Predictions for Frame {name}", fontsize=16)

    if colorbar_im is not None:
        cbar = fig.colorbar(colorbar_im, ax=axes[:, :3].ravel().tolist(),
                            orientation='horizontal', pad=0.02, shrink=0.6, aspect=30)
        cbar.set_label('Count', fontsize=12)

    out_path = out_dir / f"{name}_overlay_3x4_with_labels.png"
    fig.savefig(out_path, dpi=220)
    print(f"[PLOT] Wrote: {out_path}")

    # Display inline
    plt.show()

    return out_path

## Run

The cell below runs both stages (aggregation then plotting). Toggle `do_aggregation=False` to skip the GeoTIFF summation step if it has already been done.

In [ ]:
result_roots = [
    Path(UNET_AE_DIR), Path(UNET_PS_DIR), Path(UNET_S2_DIR),
    Path(SWIN_AE_DIR), Path(SWIN_PS_DIR), Path(SWIN_S2_DIR),
    Path(TERRAMIND_AE_DIR), Path(TERRAMIND_PS_DIR), Path(TERRAMIND_S2_DIR),
]

summed_dirs = {
    ("UNET",      "AE"): Path(UNET_AE_DIR),
    ("UNET",      "PS"): Path(UNET_PS_DIR),
    ("UNET",      "S2"): Path(UNET_S2_DIR),
    ("SWIN",      "AE"): Path(SWIN_AE_DIR),
    ("SWIN",      "PS"): Path(SWIN_PS_DIR),
    ("SWIN",      "S2"): Path(SWIN_S2_DIR),
    ("TERRAMIND", "AE"): Path(TERRAMIND_AE_DIR),
    ("TERRAMIND", "PS"): Path(TERRAMIND_PS_DIR),
    ("TERRAMIND", "S2"): Path(TERRAMIND_S2_DIR),
}

training_dirs = {
    "AE": Path(TRAINING_IMAGE_DIR_AE),
    "PS": Path(TRAINING_IMAGE_DIR_PS),
    "S2": Path(TRAINING_IMAGE_DIR_S2),
}

viz_out = Path(RESULTS_DIR) / "viz_overlays_3x4"

# --- Step 1: Aggregate GeoTIFFs (set do_aggregation=False to skip) ---
do_aggregation = True
skip_if_done   = True

if do_aggregation:
    print("=== STEP 1: SUM GEO-TIFFS ===")
    for root in tqdm(result_roots, desc="Aggregating"):
        if not root.exists():
            print(f"[WARN] Missing: {root}")
            continue
        if skip_if_done and _aggregation_done_for_names(root, VIZ_NAMES):
            print(f"[SKIP] Already done: {root / 'summed'}")
            continue
        try:
            sum_numeric_named_geotiffs_across_subfolders(
                root, output_folder_name="summed",
                require_all_subfolders=False, overwrite=True,
            )
        except Exception as ex:
            print(f"[ERROR] {root} -> {type(ex).__name__}: {ex}")
else:
    print("=== STEP 1: SKIPPED ===")

# --- Step 2: Plot ---
print("\n=== STEP 2: PLOT 3x4 ===")
for nm in tqdm(VIZ_NAMES, desc="Plotting"):
    try:
        plot_3x4_overlays_with_label_col(
            name=nm,
            training_dirs=training_dirs,
            summed_dirs=summed_dirs,
            out_dir=viz_out,
            add_contours=ADD_CONTOURS,
        )
    except Exception as ex:
        print(f"[ERROR] name={nm} -> {type(ex).__name__}: {ex}")

print("\nDone.")